# Walkie 设定：无偏置 SwiGLU MLP 与 SiLU

源码导航：[`core/ffn/swiglu.py`](../../../core/ffn/swiglu.py) 中的 `SwiGLUMLP`。

在经典架构的 Transformer (如 GPT-2) 中，FFN 常设计为扩展至 4倍维度的简单感知机：`FFN(x) = GELU(x @ W1 + b1) @ W2 + b2`。
由于目前大参数模型的工程探索（LLaMA / PaLM 此类），学术界发现使用**Gated Linear Units (门控线性单元)**中的 **SwiGLU** 变体能够同开销下具有更好的语言建模性能。Walkie 的设计同样剥离了所有 `bias` 偏置以缩减算力消耗。

### 1. SwiGLU 的双通路设计与公式

给定输入序列的隐状态 $x\in\mathbb{R}^{B\times T\times n\_embd}$，SwiGLU 设置了两条隐空间通路：

$$
\operatorname{SwiGLU}(x)= \left(\operatorname{SiLU}(x W_{gate}) \odot (x W_{up})\right) W_{down}
$$

1. **门控通路 (Gate)**: 获取放大维度后的表达 $x W_{gate}$ ，传入非线性激活函数 **SiLU**，等效于一个可以开关信息的阀门矩阵。
2. **直通通路 (Up)**: 平行映射同样倍率的特征 $x W_{up}$。
3. **点积聚合**: 两条通道的输出结构一致，利用哈达玛乘积（逐元素乘法 $\odot$）相融合。门阀起到筛选信息强度的作用。
4. **降维重构 (Down)**: 最后，乘以 $W_{down}$ 一次性压缩映射回原本输入的隐维度 $n\_embd$ 处。

由于增加了额外的投影矩阵，为了在近似的参数范围内达到原 4倍 扩容的效果，业界标准通常将隐含层特征空间的上升倍率设定为 $\frac{8}{3}n\_embd$（即参数 `d_ffn`）。

### 2. SiLU 激活分析
引入的 **SiLU (Swish)** 数学定义为：
$$
\operatorname{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}
$$
对比 ReLU (`max(0, x)`)，它不但在原点绝对平滑可导，并且在微小负值区域没有生硬的一刀切，存在平缓的下凹渗漏区，这为模型提供了更好的梯度回溯，克服死神经元的问题。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ffn.swiglu import SwiGLUMLP

In [ ]:
import matplotlib.pyplot as plt

# 构建 -5 到 5 区域内的等距标量
x_plt = torch.linspace(-5, 5, 200)

# 使用 PyTorch 中的 SiLU 以及传统的 ReLU 方法便于对比
y_silu = torch.nn.functional.silu(x_plt)
y_relu = torch.nn.functional.relu(x_plt)

plt.figure(figsize=(8, 5))
plt.plot(x_plt.numpy(), y_silu.numpy(), label='SiLU (Swish)', color='r', linewidth=2.5)
plt.plot(x_plt.numpy(), y_relu.numpy(), label='ReLU', color='b', linestyle='--', linewidth=2)

plt.axhline(0, color='black',linewidth=1.2, linestyle='dotted')
plt.axvline(0, color='black',linewidth=1.2, linestyle='dotted')
plt.title("SiLU vs. ReLU Activation Function")
plt.xlabel("Input (x)")
plt.ylabel("Output (y)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 3. 数据测试举例

In [ ]:
torch.manual_seed(0)

# 初始化 SwiGLUMLP，去除 bias
mlp = SwiGLUMLP(n_embd=128, d_ffn=256, dropout=0.0, bias=False)

# 输入测试数据。形状为 (Batch=2, Seq_len=8, n_embd=128)
x = torch.randn(2, 8, 128)
y = mlp(x)

print(f"输入特征 x 形状: {tuple(x.shape)}")
print(f"输出特征 y 形状: {tuple(y.shape)}")
assert x.shape == y.shape, "SwiGLU 必须保持输入和输出的特征维度一致！"

### 4. 对应源码展示与参数盘点
我们在下面调用原始的 `SwiGLUMLP` 对象查看参数规模。你会发现没有了任何 bias 后，这里的组成只剩下三个方正的 weight 矩阵。

In [ ]:
for name, p in mlp.named_parameters():
    print(f'{name:18s}', tuple(p.shape), p.numel())
print('total:', sum(p.numel() for p in mlp.parameters()))

其具体核心代码（参考了 LLaMA 的简洁实现并在 Walkie 中应用）为：
```python
class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd: int, d_ffn: int, dropout: float = 0.0, bias: bool = False):
        super().__init__()
        # 对应门控通路，不加偏置
        self.gate_proj = nn.Linear(n_embd, d_ffn, bias=bias)
        # 对应直通内容，不加偏置
        self.up_proj = nn.Linear(n_embd, d_ffn, bias=bias)
        # 对应降维重构，不加偏置
        self.down_proj = nn.Linear(d_ffn, n_embd, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # F.silu 将其转换为概率强度的门限
        gate = F.silu(self.gate_proj(x))
        # 线性投射的实际特征内容
        up = self.up_proj(x)
        # 逐元素乘积提取特写特征，随后做投影
        return self.dropout(self.down_proj(gate * up))
```

---

## 延伸阅读与参考资料

### 核心论文
- **GLU Variants Improve Transformer**: Shazeer, 2020. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)
- **PaLM**: Chowdhery et al., 2022. [arXiv:2204.02311](https://arxiv.org/abs/2204.02311)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)

### 工程实现
- **Hugging Face Transformers LlamaMLP**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)
- **PyTorch SiLU API**: [docs](https://pytorch.org/docs/stable/generated/torch.nn.SiLU.html)